# 🎬 TwinVision: Prompt-to-Video AI Model Comparison

**Two AI image models. One pipeline from prompt to video. Side-by-side evaluation.**

This notebook runs the full TwinVision pipeline on a Google Colab T4 GPU:

1. Generate images from a text prompt using **FLUX.1 [schnell]** and **SD 3.5 Large**
2. Assemble the images into MP4 videos using **FFmpeg** with Ken Burns zoom + crossfade
3. Score both outputs with **5 automated metrics**: CLIP Score, BRISQUE, NIQE, SSIM, LPIPS
4. Determine the **winner** via a weighted comparison report

---

| Model | HuggingFace ID | Steps | Guidance | Resolution |
|---|---|---|---|---|
| FLUX.1 [schnell] | `black-forest-labs/FLUX.1-schnell` | 4 | 0.0 | 1024×1024 |
| SD 3.5 Large | `stabilityai/stable-diffusion-3.5-large` | 28 | 7.5 | 1024×1024 |

> **Prerequisites:** A HuggingFace account with access to both gated models.

In [ ]:
# Cell 2 — GPU check
# Verify a T4 (or better) GPU is available before proceeding.
!nvidia-smi
import torch
print(f"\nPyTorch sees CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Pipeline will be extremely slow on CPU.")

In [ ]:
# Cell 3 — Install dependencies
# This takes 3-5 minutes on first run; subsequent runs use the Colab cache.
!apt-get install -y ffmpeg -q
!pip install -r requirements.txt -q
print("\nAll dependencies installed.")

In [ ]:
# Cell 4 — Upload or clone the TwinVision project files
#
# Option A: Clone from GitHub (recommended if the repo is public)
#   !git clone https://github.com/YOUR_USERNAME/TwinVision.git
#   %cd TwinVision
#
# Option B: Upload a zip from your local machine
#   from google.colab import files
#   uploaded = files.upload()          # select TwinVision.zip
#   !unzip -q TwinVision.zip
#   %cd TwinVision
#
# Option C: Mount Google Drive
#   from google.colab import drive
#   drive.mount('/content/drive')
#   %cd /content/drive/MyDrive/TwinVision

# --- uncomment ONE option above and run this cell ---

import os
print("Current directory:", os.getcwd())
print("Files:", os.listdir('.'))

In [ ]:
# Cell 5 — Set HuggingFace token
#
# Both FLUX.1-schnell and SD 3.5 Large are gated models on HuggingFace.
# You must have accepted their terms of use at:
#   https://huggingface.co/black-forest-labs/FLUX.1-schnell
#   https://huggingface.co/stabilityai/stable-diffusion-3.5-large
#
# Get your token at: https://huggingface.co/settings/tokens
# (needs at least 'read' scope)

import os

HF_TOKEN = "hf_YOUR_TOKEN_HERE"   # <-- replace with your actual token

os.environ["HF_TOKEN"] = HF_TOKEN

# Write to .env so pipeline modules can also load it via python-dotenv.
with open(".env", "w") as f:
    f.write(f"HF_TOKEN={HF_TOKEN}\n")

print("HF_TOKEN set." if HF_TOKEN != "hf_YOUR_TOKEN_HERE" else "WARNING: Replace the placeholder token before running the pipeline.")

In [ ]:
# Cell 6 — Quick verification run
# Generates 2 images per model for the first config prompt.
# Should complete in under 5 minutes on a T4.
# Check for errors here before committing to the full run.
!python run_pipeline_cli.py --test
print("\nTest run complete. Check output/ for generated files.")

In [ ]:
# Cell 7 — Full pipeline run
# Generates 7 images per model across all 5 config prompts.
# Estimated time on T4: ~25-35 min total (SD 3.5 is the bottleneck).
#
# To run with a single custom prompt instead:
#   !python run_pipeline_cli.py --prompt "Your custom prompt here"

!python run_pipeline_cli.py

# After the run, discover the output directory for display cells below.
import glob, os
job_dirs = sorted(glob.glob("output/*/results/metrics.csv"), key=os.path.getmtime)
if job_dirs:
    LATEST_JOB = job_dirs[-1].split(os.sep)[1]
    print(f"\nLatest job ID: {LATEST_JOB}")
else:
    LATEST_JOB = None
    print("No completed job found. Run the pipeline first.")

In [ ]:
# Cell 8 — Display generated images side by side
import glob
from IPython.display import Image, display
import ipywidgets as widgets

JOB = LATEST_JOB   # set automatically by Cell 7; override here if needed

flux_imgs = sorted(glob.glob(f"output/{JOB}/flux/prompt_0_img_*.png"))
sd35_imgs = sorted(glob.glob(f"output/{JOB}/sd35/prompt_0_img_*.png"))

print(f"FLUX.1 images ({len(flux_imgs)}):")
for p in flux_imgs[:4]:   # show first 4 to save space
    display(Image(p, width=300))

print(f"\nSD 3.5 images ({len(sd35_imgs)}):")
for p in sd35_imgs[:4]:
    display(Image(p, width=300))

In [ ]:
# Cell 9 — Display generated videos
import glob
from IPython.display import Video, display

JOB = LATEST_JOB

video_files = {
    "FLUX.1": f"output/{JOB}/videos/flux_prompt_0.mp4",
    "SD 3.5": f"output/{JOB}/videos/sd35_prompt_0.mp4",
    "Side-by-side": f"output/{JOB}/videos/comparison_prompt_0.mp4",
}

import os
for label, path in video_files.items():
    if os.path.exists(path):
        print(f"\n{label}:")
        display(Video(path, embed=True, width=640))
    else:
        print(f"{label}: file not found at {path}")

In [ ]:
# Cell 10 — Show metrics table
import pandas as pd
from IPython.display import display

JOB = LATEST_JOB
csv_path = f"output/{JOB}/results/metrics.csv"

df = pd.read_csv(csv_path)
print(f"Metrics table — {len(df)} rows\n")

# Pivot to a readable summary: one row per (model, metric), averaged across prompts.
summary = (
    df.groupby(["model", "metric"])["average_score"]
    .mean()
    .unstack("metric")
    .round(4)
)
display(summary)

print("\nFull raw table:")
display(df[["prompt_index", "model", "metric", "average_score", "generation_time"]])

In [ ]:
# Cell 11 — Display comparison charts
import os
from IPython.display import Image, display

JOB = LATEST_JOB

charts = {
    "Bar Chart (raw scores)": f"output/{JOB}/results/bar_chart.png",
    "Radar Chart (normalised)": f"output/{JOB}/results/radar_chart.png",
    "Image Grid": f"output/{JOB}/results/image_grid.png",
}

for title, path in charts.items():
    if os.path.exists(path):
        print(f"\n{title}:")
        display(Image(path, width=800))
    else:
        print(f"{title}: not found at {path}")

In [ ]:
# Cell 12 — Download all results as a zip archive
import os, zipfile
from google.colab import files

JOB = LATEST_JOB
results_dir = f"output/{JOB}/results"
zip_name = f"twinvision_results_{JOB}.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(f"output/{JOB}"):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, start="output")
            zf.write(filepath, arcname)

print(f"Created {zip_name}")
print(f"Contents:")
with zipfile.ZipFile(zip_name) as zf:
    for name in zf.namelist():
        print(f"  {name}")

files.download(zip_name)
print("\nDownload started.")